# 03. Conditional VAE — Synthetic Spectrum Generation

Trains a conditional VAE (CVAE) per CV fold on real tumour spectra and generates a
balanced pool of **synthetic** spectra, consumed downstream by the TSTR / TSRTR
regimens (notebook 02) and the realism evaluation (notebook 10).

For each fold the CVAE stage:
1. Trains the CVAE with **dynamic per-epoch slide capping** — a fresh
   ≤ `cv_strategy.patient_cap` spectra-per-slide subsample *every epoch* (implicit
   regularisation) — plus optional on-the-fly augmentation (within-class mixup +
   spectral shift).
2. Optionally anneals the KL (`beta`) and classification (`alpha`) loss weights.
3. Generates `models.cvae.samples_per_class` spectra per class from the trained decoder.

### Pipeline parity
Training + generation run via `pipeline.cvae_stage.cvae_stage` — the *same* code the
headless suite uses (`python -m pipeline.run_suite --stages cvae`). This notebook is
a thin driver: pick the experiment, run the stage, inspect the outputs. See
[`pipeline/README.md`](../pipeline/README.md).

> **Slide-cap semantics.** Classifier stages (notebook 02) draw the slide cap
> **once per fold**; the CVAE re-samples it **every epoch**. The config key
> `cap_strategy: "dynamic_per_epoch"` documents *this* CVAE behaviour and is not
> read by code (only `patient_cap` is). See pipeline/README.md → "Slide cap semantics".

**Inputs:** `paths.processed_file` (H5) + `paths.splits_file`.
**Outputs (per fold):**
`encoder_{fold}.weights.h5`, `decoder_{fold}.weights.h5`, `history_{fold}.csv`
under `models_cvae_dir`; `{fold}_X_synth.npy` / `{fold}_y_synth.npy` under `synthetic_dir`.
Folds whose synthetic pool + decoder weights already exist are skipped on re-run.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Project root on path (this notebook lives in notebooks/). Must precede the
# `src` / `pipeline` imports below.
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.notebooks.experiment_utils import setup_experiment
from pipeline.cvae_stage import cvae_stage

# ── Experiment selection ───────────────────────────────────────────────
CONFIG_NAME = 'exp_201'    # any configs/exp_*.yaml
EPOCHS      = None         # None -> config's models.cvae.epochs; set e.g. 2 for a quick smoke test
BACKEND     = 'local'      # 'local' (this machine) or 'modal' (GPU offload)

# setup_experiment() chdir's to the project root so the config's relative paths resolve.
config, output_dir = setup_experiment(CONFIG_NAME)
cvae_cfg = config['models']['cvae']

print(f"Experiment : {CONFIG_NAME}  ({config.get('description', '')})")
print(f"Epochs     : {EPOCHS or cvae_cfg['epochs']}  "
      f"| latent_dim={cvae_cfg['latent_dim']}  | samples/class={cvae_cfg['samples_per_class']}")
print(f"Loss wts   : {cvae_cfg['loss_weights']}")
print(f"Backend    : {BACKEND}")

## Train + generate (all folds)

Delegates to `cvae_stage`. Each fold trains the CVAE then generates its balanced synthetic pool; folds with existing outputs are skipped. (Full 70-epoch runs are slow on CPU — set `EPOCHS = 2` above for a smoke test, or `BACKEND = 'modal'` for GPU.)

In [ ]:
cvae_stage(config, output_dir, backend=BACKEND, epochs=EPOCHS)

## Verify outputs

In [ ]:
synthetic_dir = Path(config['paths']['synthetic_dir'])
model_dir     = Path(config['paths']['models_cvae_dir'])

synth_files   = sorted(synthetic_dir.glob('*_X_synth.npy'))
decoder_files = sorted(model_dir.glob('decoder_*.weights.h5'))
history_files = sorted(model_dir.glob('history_*.csv'))

print(f"Synthetic pools : {len(synth_files):>2}   ({synthetic_dir})")
print(f"Decoder weights : {len(decoder_files):>2}   ({model_dir})")
print(f"History CSVs    : {len(history_files):>2}")

if synth_files:
    X = np.load(synth_files[0])
    y = np.load(str(synth_files[0]).replace('_X_synth', '_y_synth'))
    classes, counts = np.unique(y, return_counts=True)
    print(f"\n{synth_files[0].name}: X={X.shape} | classes={dict(zip(classes.tolist(), counts.tolist()))}")

## Diagnostics — training curves & sample synthetic spectra

In [ ]:
if history_files and synth_files:
    hist = pd.read_csv(history_files[0])
    Xs   = np.load(synth_files[0])

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(hist['epoch'], hist['total_loss'], label='total', lw=2)
    for k in ['mse_loss', 'cos_loss', 'kl_loss', 'class_loss']:
        if k in hist:
            axes[0].plot(hist['epoch'], hist[k], label=k, alpha=0.7)
    axes[0].set_xlabel('epoch'); axes[0].set_ylabel('loss')
    axes[0].set_title(f"{history_files[0].stem} — training curves")
    axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

    for i in range(min(5, len(Xs))):
        axes[1].plot(Xs[i].squeeze(), alpha=0.7)
    axes[1].set_xlabel('wavenumber index'); axes[1].set_ylabel('intensity')
    axes[1].set_title('Sample synthetic spectra'); axes[1].grid(alpha=0.3)

    plt.tight_layout(); plt.show()
else:
    print('Run the stage first to produce history + synthetic files.')